## Reading Piped Commands

Before the practice — the mental model that makes long pipelines readable instead of intimidating.

**`|` (pipe) means: "take the output of the command on my left, and feed it in as input to the command on my right."** Nothing more mystical than that. Every stage is a little assembly line worker that transforms the data slightly, then passes it down the line.

**The trick to reading any long pipeline:** don't try to understand the whole thing at once — read it **left to right, one stage at a time**, and after each `|` ask yourself "what does the data look like _now_?"

Take a pipeline you already ran:

In [ ]:
dennie@HP:~/linux-tut$ sudo grep -i "error\|fail" /var/log/syslog | awk '{$1=$2=$3=$4="";print $0}' | sort | uniq -c | sort -rn | head -10
[sudo] password for dennie: 
      8     to connect to bus: No such file or directory
      8     - Process error reports when automatic reporting is enabled (timer based) was skipped because of an unmet condition check (ConditionPathExists=/var/lib/apport/autoreport).
      8     - Process error reports when automatic reporting is enabled (file watch) was skipped because of an unmet condition check (ConditionPathExists=/var/lib/apport/autoreport).
      7     to register legacy timer interrupt
      7     Underlying Hyper-V does not support order less than 9. Hypercall failed
      7     Correctable Errors collector initialized.
      5     (107) ERROR: CheckConnection: getaddrinfo() failed: -5
      3     Cold memory discard hypercall failed with status e00000005
      3     Cold memory discard hypercall failed with status 1800000005
      3     (109) ERROR: CheckConnection: getaddrinfo() failed: -5

Walk it stage by stage:

1. `grep -i "error|fail" file` → **output:** only the log lines mentioning error/fail (still full lines, timestamps and all)
2. `| awk '{...}'` → **output:** same lines, but the first 4 fields blanked out — now it's just the bare messages
3. `| sort` → **output:** same lines, now alphabetically grouped so identical messages sit next to each other
4. `| uniq -c` → **output:** each _unique_ message, once, with a count prefix
5. `| sort -rn` → **output:** same list, reordered biggest-count-first
6. `| head -10` → **output:** just the top 10 lines of that

Six tiny transformations, chained. That's the whole skill — **each command only needs to know how to do its own small job**; the pipe is what turns five simple tools into one powerful diagnostic. When you see an unfamiliar 6-stage pipeline in a doc or ticket, don't panic-read it as one blob — isolate each `|` boundary and narrate what the data looks like at each handoff, same as above.

---

Commands split into two rough categories:

- **Stdin-readers** (`grep`, `sort`, `uniq`, `awk`, `sed`, `cat`, `wc`) — these are built to receive a stream of _data_ on stdin and process it line by line. Piping into them works naturally.
- **Argument-takers** (`readlink`, `ls`, `chmod`, `rm`, `cp`, `stat`, `file`) — these expect the _filename itself_ as an argument on the command line. Piping a filename into their stdin does nothing useful — they're not looking there at all.